# Clean ice-accident records

`records` (built by `transform.py`) stacks every raw source together, so this
notebook first filters it down to just the `ice_reports` rows, then shapes
them the same way `01_clean_data.ipynb` shapes avalanche data: lower-cased
column names, a real `dato` dtype with derived `år`/`måned`/`dag`, count
columns as `int64`, `source` dropped.

Saved as `ice_reports_clean` — a page can load it with
`storage.load("ice_reports_clean")`.

In [1]:
import sys

sys.path.insert(0, "..")  # the notebook runs from 03_notebooks/

import pandas as pd

from backend import storage

storage.tables()

['ice_reports_clean', 'records', 'snow_avalanche_data', 'trend_totals']

In [2]:
records = storage.load("records")
records.head()

,Dato,Døde,Kun skadet,Skredtatte,Sted,Latitude,Longitude,Kommune,Område,Aktivitet,...,Eksposisjon,Comment,source,Gjennom isen,Vann/sted,Høyde,Fylke,Vanntype,Istype,Påvirket
0,2026-05-17,0,0.0,1.0,Rundfjellet,69.565128,19.299435,Tromsø,Troms,Topptur,...,V,En person tatt av snøskred under nedkjøring. I...,avalanche_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-17,0,0.0,1.0,Tjønnholstinden,61.443767,8.650444,Vågå,Oppland,Ukjent,...,SØ,En person tatt av skred. Ikke meldt om skade.,avalanche_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-05-16,0,0.0,1.0,Storsteinnestinden,69.673372,18.499346,Tromsø,Troms,Topptur,...,V,En person ble tatt av snøskred. Vedkommende ko...,avalanche_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-05-14,0,0.0,2.0,Jægervasstindan,69.702153,20.027998,Lyngen,Troms,Topptur,...,V,Personutløst skred hvor to personer ble tatt. ...,avalanche_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-05-01,0,2.0,7.0,Konowfjellet,78.548386,12.974510,Svalbard,Svalbard,Topptur,...,NV,Fjernutløst skred hvor syv personer ble tatt o...,avalanche_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Clean `records` into `ice_reports_clean`:

1. Keep only the `ice_reports` rows (`records` also has avalanche rows —
   mixed in by `transform.py` stacking every raw source into one table), and
   drop any column that's entirely empty once filtered (the avalanche-only
   columns).
2. Lower-case every column name.
3. Parse `dato` to a real `datetime64` dtype (ISO `YYYY-MM-DD` text), then
   derive `år`, `måned` (Norwegian month name) and `dag` (Norwegian weekday
   name) from it.
4. Cast `døde` and `gjennom isen` to `int64` — `gjennom isen` (`deltagere` in
   the source data) is missing for most records; treated as zero rather than
   unknown, same reasoning as the avalanche notebook's `kun skadet`.
5. Drop rows where `fylke` is `Svalbard` — the map page's Kartverket tiles
   only cover mainland Norway, not Svalbard or Jan Mayen, so a Svalbard
   marker would sit on a blank/unrendered part of the map. Note: `vann/sted`
   never actually contains the word "Svalbard" (it's the local water body
   name, e.g. "Mohnbukta") — `fylke` is where Svalbard shows up in this
   dataset.
6. Drop the `source` column.

In [ ]:
NORWEGIAN_MONTHS = {
    1: "Januar", 2: "Februar", 3: "Mars", 4: "April", 5: "Mai", 6: "Juni",
    7: "Juli", 8: "August", 9: "September", 10: "Oktober", 11: "November", 12: "Desember",
}
NORWEGIAN_WEEKDAYS = {
    0: "Mandag", 1: "Tirsdag", 2: "Onsdag", 3: "Torsdag", 4: "Fredag", 5: "Lørdag", 6: "Søndag",
}

clean = pd.DataFrame()

if records.empty or "source" not in records.columns or "ice_reports" not in records["source"].values:
    print("No `ice_reports` rows in `records` yet — run `make pipeline` first.")
else:
    clean = records[records["source"] == "ice_reports"].dropna(axis=1, how="all")
    clean.columns = clean.columns.str.lower()

    clean["dato"] = pd.to_datetime(clean["dato"], format="%Y-%m-%d")
    clean["år"] = clean["dato"].dt.year
    clean["måned"] = clean["dato"].dt.month.map(NORWEGIAN_MONTHS)
    clean["dag"] = clean["dato"].dt.weekday.map(NORWEGIAN_WEEKDAYS)

    count_columns = ["døde", "gjennom isen"]
    clean[count_columns] = clean[count_columns].fillna(0).astype("int64")

    # Svalbard has real accidents in the data, but the map page's Kartverket
    # tiles only cover mainland Norway (verified: Svalbard/Jan Mayen tiles
    # come back blank) — drop so every remaining marker lands on real terrain.
    # vann/sted itself never says "Svalbard" (it's a local water body name),
    # so fylke is the column that actually carries this.
    clean = clean[clean["fylke"] != "Svalbard"]

    clean = clean.drop(columns="source")

    storage.save("ice_reports_clean", clean)

clean.head()